# Book Recommendation Engine

Create a collaborative-filtering engine that recommends books from reader-rating behaviour.

**Portfolio category:** Recommendation

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Transparent demonstration data

In [ ]:
categories = ["data", "fiction", "history", "business", "science"]
items = pd.DataFrame({
    "item_id": np.arange(30),
    "title": [f"Book {i + 1:02d}" for i in range(30)],
    "category": np.repeat(categories, 6),
})
user_preferences = rng.dirichlet(np.ones(len(categories)) * 0.8, size=90)
rows = []
for user_id, preferences in enumerate(user_preferences):
    chosen = rng.choice(items["item_id"], size=rng.integers(8, 18), replace=False)
    for item_id in chosen:
        category_index = categories.index(items.loc[item_id, "category"])
        latent = 1 + 4 * preferences[category_index]
        rating = int(np.clip(np.rint(latent + rng.normal(0, 0.7)), 1, 5))
        rows.append((user_id, item_id, rating))
ratings = pd.DataFrame(rows, columns=["user_id", "item_id", "rating"])
interactions = ratings.merge(items, on="item_id")
display(interactions.head())

## 3. Data quality and interaction density

In [ ]:
density = len(ratings) / (ratings["user_id"].nunique() * ratings["item_id"].nunique())
display(pd.Series({
    "ratings": len(ratings),
    "users": ratings["user_id"].nunique(),
    "items": ratings["item_id"].nunique(),
    "interaction_density": density,
    "duplicate_pairs": ratings.duplicated(["user_id", "item_id"]).sum(),
}).to_frame("value"))

## 4. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=interactions, x="rating", ax=axes[0], color="#2563eb")
interactions.groupby("category")["rating"].mean().sort_values().plot.barh(
    ax=axes[1], color="#10b981"
)
axes[0].set_title("Rating distribution")
axes[1].set_title("Average rating by category")
plt.tight_layout()

## 5. Item-to-item collaborative filtering

In [ ]:
popularity = ratings.groupby("item_id").size()
eligible = popularity[popularity >= popularity.quantile(0.25)].index
matrix = ratings[ratings["item_id"].isin(eligible)].pivot_table(
    index="item_id", columns="user_id", values="rating"
).fillna(0)
similarity = pd.DataFrame(
    cosine_similarity(matrix), index=matrix.index, columns=matrix.index
)
np.fill_diagonal(similarity.values, 0)

def recommend(item_id, n=6):
    scores = similarity.loc[item_id].nlargest(n)
    result = items.set_index("item_id").loc[scores.index].copy()
    result["similarity"] = scores.values
    return result.reset_index()

seed_item = int(popularity.idxmax())
recommendations = recommend(seed_item)
display(items.query("item_id == @seed_item"))
display(recommendations)

## 6. Coverage and diversity checks

In [ ]:
sample_items = list(similarity.index[:20])
recs = pd.concat([recommend(item_id, 5).assign(seed=item_id) for item_id in sample_items])
coverage = recs["item_id"].nunique() / len(items)
diversity = recs.groupby("seed")["category"].nunique().mean()
display(pd.DataFrame({
    "metric": ["catalogue coverage", "mean category diversity"],
    "value": [coverage, diversity],
}))

## 7. Similarity diagnostics

In [ ]:
subset = similarity.iloc[:12, :12]
sns.heatmap(subset, cmap="Blues")
plt.title("Item similarity sample")
plt.tight_layout()

## 8. Recommendation explanation

In [ ]:
seed_category = items.set_index("item_id").loc[seed_item, "category"]
recommendations["same_category_as_seed"] = recommendations["category"].eq(seed_category)
display(recommendations)

## 9. Key findings

Evaluate relevance, coverage and diversity together; a high similarity score alone does not prove a useful recommender.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For book recommendation engine,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.